The Scenario: You are given three separate datasets: "User Demographics", "Track Metadata" (artist, genre, duration), and "Listening History" (who listened to what, and when).
Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. Then, use pd.merge() to join the 'Listening History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join.
Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.
Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.
Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.
Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.

In [2]:
import pandas as pd
q1=pd.read_csv('listening_history_q1.csv')
q2=pd.read_csv('listening_history_q2.csv')
listening_his=pd.concat([q1,q2],ignore_index=True)

track_metadata = pd.read_csv('track_metadata.csv')
user_demo=pd.read_csv("user_demographics.csv")
print(listening_his.head())
listening_with_track=pd.merge(
    listening_his, track_metadata, on='trk_ref_id', how='left'
)
print(listening_with_track.head())
listening_with_track['timestamp']=(
    listening_with_track['length_in_ms']/60000
)

listening_with_track=listening_with_track.rename(columns={
    'user_id':'User_ID',
    'Genre':'Genre',
    'timestamp':'Timestamp'
})

listening_with_track=listening_with_track.drop(
    columns=['track_id'],
    errors='ignore'
)
print(listening_with_track.head())

listening_with_track['timestamp']=pd.to_datetime(
    listening_with_track['timestamp']
)
def time_category(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'
listening_with_track['Time_of_day']=(
    listening_with_track['timestamp'].apply(lambda x:time_category(x.hour))
)
print(listening_with_track[['timestamp','Time_of_day']].head())
genre_stats=(
    listening_with_track.groupby('Genre').agg({
        'Minutes_listened':'sum',
        'duration_ms':'mean',
        'User_ID':'nunique'
    }).rename(columns={
        'Minutes_listened':'Total_minutes_listened',
        'duration_ms':'Average_duration_ms',
        'User_ID':'Unique_listeners'
    })
)
print(genre_stats)
final_df=pd.merge(
    listening_with_track,user_demo,on='User_ID',how='left'
)
pivot_table=pd.pivot_table(
    final_df,values='trk_ref_id',
    index='age_group',columns='Genre',aggfunc='count',fill_value=0
)
print(pivot_table)

   listen_id  usr_ref_id  trk_ref_id            timestamp
0          1         167         146  2025-01-23 22:19:45
1          3          90         105  2025-01-06 15:32:21
2          5          39         119  2025-02-12 11:12:04
3          6         126         149  2025-03-05 19:52:42
4          8         173         101   2025-01-04 5:56:14


KeyError: 'trk_ref_id'